# Cross-Engine Surge & INP Verification

Same **checked-in reference** pattern as [`quickstart_notebook.ipynb`](quickstart_notebook.ipynb) (R-THYM JSON/CSV), but for surge and EPANET topics:

| Engine | Artifact | This notebook |
|--------|----------|---------------|
| **R-THYM** | `tests/R-THYM_Joukowsky_*`, `tests/R-THYM_MOC_*` | See quickstart / long-pipe notebooks |
| **TSNet** | `tests/TSNet_Standpipe_B8_Verification.json`, `tests/TSNet_Standpipe_B8_Traces.csv` | §2 overlay (Appendix B.8.5) |
| **EPANET** | live steady-state via **wntr** on `tests/networks/complex_topology.inp` | §3 pre-trip bars |

Requires **`pip install 'rthym-moc[inp]'`** for §3.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rthym_moc
from _verification_notebook_setup import bootstrap_verification_notebook

REPO_ROOT, TESTS_DIR = bootstrap_verification_notebook(require_wntr=True)
print(f"Repository root: {REPO_ROOT}")
print(f"rthym_moc: {getattr(rthym_moc, '__version__', 'unknown')}")

## 2. TSNet standpipe B.8 — checked-in trace vs RTHYM-MOC

In [ ]:
from cross_engine_verification_utils import (
    TSNET_PEAK_DIFF_TOL_FT,
    TSNET_RMS_TOL_FT,
    evaluate_tsnet_standpipe_cross_engine,
    load_tsnet_standpipe_verification,
)

ref = load_tsnet_standpipe_verification()
print(f"TSNet reference: {ref['source']}")
print(f"  peak={ref['peak_head_ft']:.2f} ft  archived RMS={ref.get('rms_head_ft', 'n/a')}")

from pathlib import Path

res, t_ref, h_ref, m = evaluate_tsnet_standpipe_cross_engine()
t_sim = np.asarray(res["time"], dtype=float)
h_sim = np.asarray(res["node_head"]["SP1"], dtype=float)
mask = t_sim <= m.compare_window_s
fig, ax = plt.subplots(figsize=(12, 5))
csv_path = TESTS_DIR / "TSNet_Standpipe_B8_Traces.csv"
if csv_path.is_file() and t_ref.size:
    ax.plot(t_ref, h_ref, "--", color="gray", linewidth=1.2, label="TSNet reference (J1)")
else:
    ax.axhline(ref["peak_head_ft"], color="gray", linestyle="--", label="TSNet peak (archived)")
    ax.axhline(ref["steady_state_head_ft"], color="silver", linestyle=":", label="TSNet SS (archived)")
    print("Tip: run scripts/export_tsnet_standpipe_reference.py to add the full TSNet trace CSV.")
ax.plot(t_sim[mask], h_sim[mask], color="teal", linewidth=1.8, label="RTHYM-MOC (SP1)")
ax.set_xlim(0, m.compare_window_s)
ax.set_ylabel("Head (ft)")
ax.set_title(f"B.8 standpipe — peak diff {m.peak_diff_ft:.2f} ft [{ 'PASS' if m.passed_peak else 'FAIL' }]")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()
print(f"Peak PASS={m.passed_peak} (tol {TSNET_PEAK_DIFF_TOL_FT} ft)")

## 3. EPANET steady state vs MOC (`complex_topology.inp`)

Full walkthrough: [`epanet_import_verification.ipynb`](epanet_import_verification.ipynb).

In [ ]:
from cross_engine_verification_utils import evaluate_epanet_complex_topology_pretrip

bundle, summary = evaluate_epanet_complex_topology_pretrip()
nodes = [x.id for x in bundle.pretrip_head_metrics]
errs = [x.error for x in bundle.pretrip_head_metrics]
colors = ["seagreen" if x.passed else "crimson" for x in bundle.pretrip_head_metrics]
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(nodes)), errs, color=colors)
ax.axhline(0.5, color="gray", linestyle="--", label="tol 0.5 ft")
ax.set_xticks(range(len(nodes)), nodes, rotation=45, ha="right")
ax.set_ylabel("|MOC − EPANET| mean head (ft)")
ax.set_title(f"Pre-trip heads {summary.head_passed}/{summary.head_total} PASS")
ax.legend()
fig.tight_layout()
plt.show()
print(f"Flows {summary.flow_passed}/{summary.flow_total} PASS - overall {summary.passed}")